This notebook is designed to be placed in the root of your repository. It replicates the core logic of your scripts (extract_png_from_fits.py and combiner.py) inside interactive cells, allowing users to tweak parameters in real-time and immediately visualize the mathematical impact on their astronomical images.
🪐 Celestial Farbkasten: The Interactive Workbench

Welcome, Astro-Imager.
This notebook allows you to interactively tune the parameters that drive the Celestial-Farbkasten pipeline. Instead of editing JSON files and re-running scripts blindly, we will step through the math of extracting scientific data and recombining it into art.
🛠️ Prerequisites

Ensure you have the environment installed:

In [ ]:
!pip install -r requirements.txt
!pip install matplotlib

1. Setup & Imports

We import the necessary libraries. Note that we rely heavily on astropy.visualization for the scientific stretching of data.

In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
from astropy.io import fits
from astropy.visualization import ImageNormalize, AsinhStretch, LogStretch, LinearStretch, SqrtStretch, ZScaleInterval, MinMaxInterval, AsymmetricPercentileInterval
import cv2


# Helper to display images in the notebook (Matplotlib uses RGB, OpenCV uses BGR)
def show_image(img_array, title="Image", cmap=None):
    plt.figure(figsize=(10, 10))
    if len(img_array.shape) == 3:
        # Convert BGR to RGB for correct display
        plt.imshow(cv2.cvtColor(img_array, cv2.COLOR_BGR2RGB))
    else:
        plt.imshow(img_array, cmap=cmap or "gray")
    plt.title(title)
    plt.axis("off")
    plt.show()

Part 1: The "Negative" (Extraction)

The goal here is to convert high-dynamic-range (32-bit float) FITS data into a flat 8-bit image (0-255) without losing the faint nebulosity or blowing out the bright stars.
Step 1.1: Load Your Data

Replace the path below with a FITS file you have downloaded via download_from_csv.py.

In [ ]:
# USER: Set your FITS file path here
FITS_PATH = "outputs/download/NGC-628/jw02107-o039_t018_miri_f770w_i2d.fits"

# Loading the data (HDU 1 usually contains the SCI data for JWST)
with fits.open(FITS_PATH) as hdul:
    raw_data = hdul[1].data
    # Clean NaNs (empty pixels) by replacing them with 0
    raw_data = np.nan_to_num(raw_data, nan=0.0)

print(f"Loaded data shape: {raw_data.shape}")
print(f"Min value: {np.min(raw_data):.4f}, Max value: {np.max(raw_data):.4f}")

Step 1.2: Define Extraction Parameters

This is the control panel. These variables mirror the keys found in your configs/extract/*.json files.

    Stretch: How we compress the massive brightness range.

        AsinhStretch / LogStretch: Best for "Both" (nebula + stars). Lifts shadows.

        LinearStretch: Best for "High Intensity" (star cores only).

    Percentiles: The "Levels" adjustment.

        percentile_black: Higher = darker background, less noise.

        percentile_white: Lower = brighter image, more clipped highlights.

In [ ]:
# --- INTERACTIVE PARAMETERS ---
# 1. The Stretch Function
STRETCH_CHOICE = "AsinhStretch"  # Options: "AsinhStretch", "LogStretch", "LinearStretch", "SqrtStretch"

# 2. The Interval (Auto-range)
INTERVAL_CHOICE = "ZScale"  # Options: "ZScale", "MinMax"

# 3. Fine-Tuning the Histogram (The "Levels")
PERCENTILE_BLACK = 20.0  # Cutoff for the darkest pixels (0-100)
PERCENTILE_WHITE = 99.5  # Cutoff for the brightest pixels (0-100)

# 4. Out-of-bounds Handling
REPLACE_BELOW_BLACK = 0  # Pixel value (0-255) for data < black point
REPLACE_ABOVE_WHITE = 255  # Pixel value (0-255) for data > white point
BACKGROUND_COLOR = 0  # Color for NaN/empty regions

Step 1.3: Run the Extraction Math

This cell replicates the logic inside src/extract_png_from_fits.py. We normalize the data to 0.0-1.0 and then scale it to 0-255.

In [ ]:
def process_extraction(data, stretch_name, interval_name, p_black, p_white):
    # Map string names to Astropy functions
    stretch_map = {"AsinhStretch": AsinhStretch(), "LogStretch": LogStretch(), "LinearStretch": LinearStretch(), "SqrtStretch": SqrtStretch()}
    interval_map = {"ZScale": ZScaleInterval(), "MinMax": MinMaxInterval()}

    # 1. Normalize (Map raw flux -> 0.0 to 1.0)
    norm = ImageNormalize(data, interval=interval_map[interval_name], stretch=stretch_map[stretch_name])
    normalized_data = norm(data)

    # 2. Calculate Cutoff Levels
    # We use nanpercentile to ignore empty regions
    black_level = np.nanpercentile(normalized_data, p_black)
    white_level = np.nanpercentile(normalized_data, p_white)

    # 3. Rescale to 0-255 (The math of contrast)
    # Formula: (value - black) / (white - black) * 255
    denominator = white_level - black_level
    if denominator == 0:
        denominator = 1e-5  # prevent divide by zero

    rescaled = (normalized_data - black_level) / denominator * 255

    # 4. Clip and Cast
    rescaled = np.clip(rescaled, 0, 255)

    # 5. Apply Replacements (Masking)
    final_img = rescaled.astype(np.uint8)

    # Apply user overrides for clipped data
    final_img[normalized_data < black_level] = REPLACE_BELOW_BLACK
    final_img[normalized_data > white_level] = REPLACE_ABOVE_WHITE

    return final_img


# Execute
extracted_image = process_extraction(raw_data, STRETCH_CHOICE, INTERVAL_CHOICE, PERCENTILE_BLACK, PERCENTILE_WHITE)

# Visualize
show_image(extracted_image, title=f"Extracted: {STRETCH_CHOICE} | B:{PERCENTILE_BLACK}% W:{PERCENTILE_WHITE}%")

Part 2: The Palette (Recombination)

Now that we have an 8-bit grayscale image (or multiple), we assign them colors. This mimics src/combiner.py.
Step 2.1: Define Your Palette

We use the standard colors defined in the repository.

In [ ]:
# The "Crayons" of Celestial Farbkasten
COLORS = {
    "$CosmicGold": (20, 150, 255),  # BGR Format
    "$DeepSpaceBlue": (100, 30, 20),
    "$NebulaMagenta": (200, 40, 180),
    "$CyanGas": (220, 200, 0),
    "$Starlight": (150, 223, 255),
    "$RoyalVoid": (130, 0, 75),
    "$OxygenTeal": (160, 180, 40),
    "$StellarCrimson": (0, 50, 255),
    # ... add your own custom colors here: (Blue, Green, Red)
}

Step 2.2: Define the Layer Stack

This list mimics the images array in your combine JSON files.

    intensity_image: The grayscale image we generated in Part 1. (In a real scenario, you'd load multiple different files here).

    color: Key from the dictionary above.

    factor: Brightness multiplier.

In [ ]:
# --- INTERACTIVE COMPOSITION ---
# For this demo, we will use the *same* extracted image for both layers
# to demonstrate tinting, but usually you would use different filters (e.g., F770W vs F1130W).

LAYER_CONFIG = [
    {
        "image": extracted_image,  # The data source
        "color": "$DeepSpaceBlue",  # The Tint
        "factor": 1.0,  # Base brightness
    },
    {
        "image": extracted_image,  # Using the same image for demo
        "color": "$Starlight",  # Highlighting bright spots
        "factor": 0.5,  # Subtle overlay
    },
]

# Post-Processing
SATURATION_BOOST = 1.5  # >1.0 makes colors pop
CONTRAST_BOOST = 1.2  # >1.0 deepens blacks

Step 2.3: Run the Recombination Math

This function performs the weighted summation and HSV post-processing found in combiner.py.

In [ ]:
def process_combination(layers, sat_scale, cont_scale):
    # Initialize blank canvas (Float32 for accurate math)
    h, w = layers[0]["image"].shape
    combined_canvas = np.zeros((h, w, 3), dtype=np.float32)

    for layer in layers:
        gray = layer["image"]
        color_key = layer["color"]
        factor = layer["factor"]

        # Get BGR values
        if isinstance(color_key, str):
            bgr = np.array(COLORS[color_key], dtype=np.float32)
        else:
            bgr = np.array(color_key, dtype=np.float32)

        # 1. Normalize Gray to 0.0-1.0
        norm_gray = gray.astype(np.float32) / 255.0

        # 2. Tint: Multiply (H, W, 1) by (3,) color vector
        # This broadasts the color across the intensity map
        tinted_layer = norm_gray[:, :, np.newaxis] * bgr * factor

        # 3. Accumulate (Linear Add)
        combined_canvas += tinted_layer

    # 4. Clip to valid range
    combined_canvas = np.clip(combined_canvas, 0, 255).astype(np.uint8)

    # 5. Post-Process (HSV Shift)
    # Convert BGR -> HSV
    hsv = cv2.cvtColor(combined_canvas, cv2.COLOR_BGR2HSV).astype(np.float32)
    h, s, v = cv2.split(hsv)

    # Scale Saturation and Value (Contrast)
    s = np.clip(s * sat_scale, 0, 255)
    v = np.clip(v * cont_scale, 0, 255)

    # Merge and convert back to BGR
    final_hsv = cv2.merge([h, s, v])
    final_img = cv2.cvtColor(final_hsv.astype(np.uint8), cv2.COLOR_HSV2BGR)

    return final_img


# Execute
final_art = process_combination(LAYER_CONFIG, SATURATION_BOOST, CONTRAST_BOOST)

# Visualize
show_image(final_art, title="Final False-Color Composite")

Summary of Concepts

    Normalization (extract): We map scientific numbers (flux) to a percentage (0-1).

    Stretching (extract): We curve that line so we can see both faint dust and bright stars.

    Tinting (combine): We turn brightness into color intensity. Dark pixels stay black; bright pixels take on the assigned color.

    Accumulation (combine): We add the light from different filters together. This works just like light in the real world (additive color mixing).